# Prepare MIMIC-IV Data

Preparing the MIMIC-IV data consists of
* adding restricted information from the MIMIC-IV clinical database to `data/mimic4_samples_pre.h5`, and
* creating `data/mimic4_signals.h5` from MIMIC-IV-ECG. 

In [ ]:
import h5py
import lab_ecg
import numpy as np
import pandas as pd
import tqdm
import wfdb

In [ ]:
samples = pd.read_hdf("data/mimic4_samples_pre.h5", key="table")

### Step 0: Download MIMIC-IV

Access to the MIMIC-IV clinical database is restricted. After being granted access, you can download it from [PhysioNet](https://physionet.org/content/mimiciv/2.2/).

In [ ]:
mimic4_root = (
    ...
)  # set to root directory of your local copy of MIMIC-IV; must contain "hosp/patients.csv" and "hosp/labevents.csv"

In [ ]:
mimic4_root = "data_restricted/mimic4"

### Step 1: Add Patient Demographics: Age & Gender

In [ ]:
patients = pd.read_csv(mimic4_root + "/hosp/patients.csv")

In [ ]:
patients["male"] = (patients["gender"] == "M").astype(np.float32)

In [ ]:
samples = samples.join(
    patients[["subject_id", "male", "anchor_age", "anchor_year"]].set_index("subject_id"), on="subject_id"
)

In [ ]:
samples["age"] = samples["anchor_age"] + samples["ecg_time"].dt.year - samples["anchor_year"]

In [ ]:
samples.drop(["anchor_age", "anchor_year"], axis=1, inplace=True)

In [ ]:
(1 - samples["male"]).sum()

In [ ]:
samples["age"].describe()

### Step 2: Add Laboratory Measurements

In [ ]:
itemids = [i for v in lab_ecg.MIMIC_LAB_MAPPING.values() for i in v]
len(itemids)

In [ ]:
unique_subjects = samples["subject_id"].unique()
len(unique_subjects)

In [ ]:
# labevents.csv is huge => load iteratively
iter_csv = pd.read_csv(
    mimic4_root + "/hosp/labevents.csv",
    parse_dates=["charttime"],
    iterator=True,
    chunksize=1000000,
    dtype={"valuenum": np.float32},
)
lab = pd.concat(
    [
        chunk.loc[
            chunk["subject_id"].isin(unique_subjects) & chunk["itemid"].isin(itemids),
            ["subject_id", "itemid", "specimen_id", "charttime", "valuenum"],
        ]
        for chunk in tqdm.tqdm(iter_csv)
    ],
    axis=0,
)

In [ ]:
# we also need specimen types
iter_csv = pd.read_csv(mimic4_root + "/hosp/labevents.csv", iterator=True, chunksize=1000000)
specimen_type = pd.concat(
    [
        chunk.loc[chunk["subject_id"].isin(unique_subjects) & (chunk["itemid"] == 52033), ["specimen_id", "value"]]
        for chunk in tqdm.tqdm(iter_csv)
    ],
    axis=0,
)

In [ ]:
# take median of concurrent values
lab = (
    lab.groupby(["subject_id", "charttime", "itemid"])["specimen_id"]
    .first()
    .to_frame("specimen_id")
    .join(lab.groupby(["subject_id", "charttime", "itemid"])["valuenum"].median().to_frame("valuenum"))
)
lab.reset_index(inplace=True)

In [ ]:
specimen_type.set_index("specimen_id", inplace=True)

In [ ]:
lab = lab_ecg.validate_lab(lab)

In [ ]:
df = lab_ecg.assign_labs_to_records(
    lab, samples, specimen_type=specimen_type, temporal_dist_minutes=30, records_time_col="ecg_time"
)

In [ ]:
samples = samples.join(df)

### Step 3: Save

In [ ]:
samples.to_hdf("data/mimic4_samples.h5", key="table", complevel=9, mode="w")

### Step 4: Descriptive Statistics: Parameter/Threshold Counts & Base Rates

In [ ]:
lst = []
for param, threshold, pretty in lab_ecg.TARGETS:
    for split in ("train", "val", "test"):
        mask = (samples[param + "_split"] == split) & samples[param].notna()
        if threshold.startswith("<"):
            br = (samples.loc[mask, param] < float(threshold[1:])).mean()
        else:
            br = (samples.loc[mask, param] > float(threshold[1:])).mean()
        lst.append((split, pretty, f"{mask.sum():d} ({br * 100:.2f}%)"))

In [ ]:
pd.DataFrame(lst).set_index([1, 0])[2].unstack().sort_index().reindex(["train", "val", "test"], axis=1)

### Step 5: MIMIC-IV-ECG Waveforms

In contrast to the MIMIC-IV clinical database, MIMIC-IV-ECG can be freely downloaded from [PhysioNet](https://physionet.org/content/mimic-iv-ecg/1.0/).

Here, we store the relevant ECG recordings in an HDF5 file for efficient access during model training, analogous to `data/mimic3_signals.h5`.

In [ ]:
mimic4_ecg_root = ...  # set to root directory of your local copy of MIMIC-IV-ECG; must contain folder "files"

In [ ]:
with h5py.File("data/mimic4_signals.h5", mode="a") as h5:
    keys = set(h5)
    for i in tqdm.tqdm(range(len(samples))):
        try:
            study_id = samples.index[i]
            key = samples["sample_id"].iloc[i]
            if key not in keys:
                subject_id = str(samples["subject_id"].iloc[i])
                path = mimic4_ecg_root + f"/files/p{subject_id[:4]}/p{subject_id}/s{study_id}/{study_id}"
                rec = wfdb.rdrecord(path)

                # sanity checks
                assert rec.sig_name == ["I", "II", "III", "aVR", "aVF", "aVL", "V1", "V2", "V3", "V4", "V5", "V6"]
                assert rec.sig_len == 5000
                assert rec.fs == 500
                assert np.abs(pd.Timestamp(rec.base_datetime) - samples["ecg_time"].iloc[i]) < pd.Timedelta(1, unit="m")

                h5[key] = rec.p_signal.astype(np.float32).T
        except Exception as ex:  # noqa: BLE001
            print(samples.index[i], ex)